In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

path = "donnees/"
accidents = pd.read_csv(path + "accidents_2024.csv", sep=";", encoding="latin-1", low_memory=False)
donnees_brute = accidents.copy()

In [2]:
accidents.info()

<class 'pandas.DataFrame'>
RangeIndex: 54402 entries, 0 to 54401
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Num_Acc                    54402 non-null  str  
 1   Département                54402 non-null  str  
 2   Agglomération              54402 non-null  str  
 3   Luminosité                 54402 non-null  str  
 4   Météo (conditions atmos.)  54391 non-null  str  
 5   Type de collision          54401 non-null  str  
 6   Catégorie de route         54402 non-null  str  
 7   Régime de circulation      54402 non-null  str  
 8   Nombre de voies            54402 non-null  str  
 9   État de la surface         54402 non-null  str  
 10  Infrastructure             54402 non-null  str  
 11  Situation de laccident    54402 non-null  str  
 12  Vitesse max autorisée      54402 non-null  str  
 13  Gravité (label)            54402 non-null  str  
dtypes: str(14)
memory usage: 5.8 MB


In [3]:
accidents.shape

(54402, 14)

In [4]:
accidents.head(5)

,Num_Acc,Département,Agglomération,Luminosité,Météo (conditions atmos.),Type de collision,Catégorie de route,Régime de circulation,Nombre de voies,État de la surface,Infrastructure,Situation de laccident,Vitesse max autorisée,Gravité (label)
0,"2,02E+11",70,Hors agglomération,Crépuscule / aube,Brouillard / fumée,2 véhicules - frontale,Route départementale,Bidirectionnelle,2 voie(s),Normale,Aucun,Sur chaussée,90 km/h,Blessé hospitalisé
1,"2,02E+11",21,En agglomération,Plein jour,Temps éblouissant,Autre collision,Voie communale,Bidirectionnelle,2 voie(s),Autre,Aucun,Sur chaussée,30 km/h,Blessé hospitalisé
2,"2,02E+11",15,Hors agglomération,Crépuscule / aube,Normale,Autre collision,Voie communale,Bidirectionnelle,2 voie(s),Normale,Aucun,Sur accotement,50 km/h,Blessé hospitalisé
3,"2,02E+11",14,En agglomération,Plein jour,Temps éblouissant,2 véhicules - par le côté,Voie communale,Bidirectionnelle,4 voie(s),Normale,Autres,Sur chaussée,50 km/h,Blessé léger
4,"2,02E+11",13,Hors agglomération,Nuit avec éclairage public allumé,Pluie légère,3 véhicules et + - collisions multiples,Autoroute,Bidirectionnelle,4 voie(s),Mouillée,Aucun,Sur chaussée,50 km/h,Blessé léger


In [5]:
accidents.describe(include="all")

,Num_Acc,Département,Agglomération,Luminosité,Météo (conditions atmos.),Type de collision,Catégorie de route,Régime de circulation,Nombre de voies,État de la surface,Infrastructure,Situation de laccident,Vitesse max autorisée,Gravité (label)
count,54402,54402,54402,54402,54391,54401,54402,54402,54402,54402,54402,54402,54402,54402
unique,1,107,2,5,9,8,8,5,14,10,11,7,35,3
top,"2,02E+11",75,En agglomération,Plein jour,Normale,2 véhicules - par le côté,Voie communale,Bidirectionnelle,2 voie(s),Normale,Aucun,Sur chaussée,50 km/h,Blessé léger
freq,54402,4191,34010,35580,41791,16371,22910,33576,35076,42418,45698,44782,24497,34744


In [6]:
valeurs_manquantes = accidents.isnull().sum()
print(valeurs_manquantes)

Num_Acc                       0
Département                   0
Agglomération                 0
Luminosité                    0
Météo (conditions atmos.)    11
Type de collision             1
Catégorie de route            0
Régime de circulation         0
Nombre de voies               0
État de la surface            0
Infrastructure                0
Situation de laccident       0
Vitesse max autorisée         0
Gravité (label)               0
dtype: int64


In [7]:
accidents.nunique()

Num_Acc                        1
Département                  107
Agglomération                  2
Luminosité                     5
Météo (conditions atmos.)      9
Type de collision              8
Catégorie de route             8
Régime de circulation          5
Nombre de voies               14
État de la surface            10
Infrastructure                11
Situation de laccident        7
Vitesse max autorisée         35
Gravité (label)                3
dtype: int64

In [8]:
gravite = accidents["Gravité (label)"].value_counts()
print(gravite)

Gravité (label)
Blessé léger          34744
Blessé hospitalisé    16432
Tué                    3226
Name: count, dtype: int64


In [9]:
accidents = accidents.drop(columns=["Num_Acc"])
print("Colonne 'Num_Acc' supprimée (identifiant non prédictif).")
print(f"Shape : {accidents.shape}")

Colonne 'Num_Acc' supprimée (identifiant non prédictif).
Shape : (54402, 13)


In [10]:
accidents["Météo (conditions atmos.)"] = accidents["Météo (conditions atmos.)"].fillna(accidents["Météo (conditions atmos.)"].mode()[0]) 

In [11]:
accidents = accidents.dropna(subset=["Type de collision"])

In [12]:
print(accidents["Météo (conditions atmos.)"].isna().sum())
print(accidents["Type de collision"].isna().sum())

0
0


In [13]:
accidents.shape

(54401, 13)

In [14]:
valeurs_vitesse = accidents["Vitesse max autorisée"].value_counts()
nb_voie = accidents["Nombre de voies"].value_counts()

In [15]:
print(valeurs_vitesse)

Vitesse max autorisée
50 km/h          24497
30 km/h           9373
80 km/h           7217
90 km/h           5054
70 km/h           3879
110 km/h          1788
130 km/h          1007
Non renseigné      740
20 km/h            233
10 km/h            152
60 km/h            118
40 km/h             86
15 km/h             71
25 km/h             54
5 km/h              35
45 km/h             17
1 km/h              16
100 km/h            16
500 km/h            12
6 km/h               8
2 km/h               7
3 km/h               5
300 km/h             2
35 km/h              2
900 km/h             2
75 km/h              1
85 km/h              1
95 km/h              1
800 km/h             1
55 km/h              1
140 km/h             1
4 km/h               1
700 km/h             1
16 km/h              1
301 km/h             1
Name: count, dtype: int64


In [16]:
print(nb_voie)

Nombre de voies
2 voie(s)               35076
1 voie(s)                5965
4 voie(s)                4754
3 voie(s)                4510
Non renseigné            2218
6 voie(s)                 959
5 voie(s)                 536
8 voie(s)                 223
7 voie(s)                  57
#VALEURMULTI voie(s)       46
10 voie(s)                 32
9 voie(s)                  14
12 voie(s)                 10
11 voie(s)                  1
Name: count, dtype: int64


In [17]:
accidents["Vitesse_num"] = accidents["Vitesse max autorisée"].str.extract(r'(\d+)')
accidents["Vitesse_num"] = pd.to_numeric(accidents["Vitesse_num"])

In [18]:
stats_descrip = accidents["Vitesse_num"].describe()
valeurs = accidents["Vitesse_num"].value_counts()

In [19]:
print(stats_descrip)

count    53661.000000
mean        59.072119
std         25.244885
min          1.000000
25%         50.000000
50%         50.000000
75%         80.000000
max        900.000000
Name: Vitesse_num, dtype: float64


In [20]:
print(valeurs)

Vitesse_num
50.0     24497
30.0      9373
80.0      7217
90.0      5054
70.0      3879
110.0     1788
130.0     1007
20.0       233
10.0       152
60.0       118
40.0        86
15.0        71
25.0        54
5.0         35
45.0        17
1.0         16
100.0       16
500.0       12
6.0          8
2.0          7
3.0          5
300.0        2
35.0         2
900.0        2
75.0         1
85.0         1
95.0         1
800.0        1
55.0         1
140.0        1
4.0          1
700.0        1
16.0         1
301.0        1
Name: count, dtype: int64


In [21]:
valeurs_aberrantes = accidents["Vitesse_num"] > 130
accidents.loc[valeurs_aberrantes, "Vitesse_num"] = np.nan

In [22]:
accidents["Vitesse_num"].describe()

count    53641.000000
mean        58.901363
std         23.374811
min          1.000000
25%         50.000000
50%         50.000000
75%         80.000000
max        130.000000
Name: Vitesse_num, dtype: float64

In [23]:
accidents["Vitesse_num"] = accidents["Vitesse_num"].fillna(accidents["Vitesse_num"].median()) 

In [24]:
accidents["Vitesse_num"].describe()

count    54401.000000
mean        58.777008
std         23.234457
min          1.000000
25%         50.000000
50%         50.000000
75%         80.000000
max        130.000000
Name: Vitesse_num, dtype: float64

In [25]:
accidents["Voies_num"] = accidents["Nombre de voies"].str.extract(r'(\d+)')
accidents["Voies_num"] = pd.to_numeric(accidents["Voies_num"])

In [26]:
stats_descrip_voies = accidents["Voies_num"].describe()
valeurs_voies = accidents["Voies_num"].value_counts()

In [27]:
print(stats_descrip_voies)

count    52137.000000
mean         2.298886
std          1.057187
min          1.000000
25%          2.000000
50%          2.000000
75%          2.000000
max         12.000000
Name: Voies_num, dtype: float64


In [28]:
print(valeurs_voies)

Voies_num
2.0     35076
1.0      5965
4.0      4754
3.0      4510
6.0       959
5.0       536
8.0       223
7.0        57
10.0       32
9.0        14
12.0       10
11.0        1
Name: count, dtype: int64


In [29]:
accidents["Voies_num"] = accidents["Voies_num"].fillna(accidents["Voies_num"].median()) 

In [30]:
accidents["Voies_num"].describe()

count    54401.000000
mean         2.286447
std          1.036674
min          1.000000
25%          2.000000
50%          2.000000
75%          2.000000
max         12.000000
Name: Voies_num, dtype: float64

In [31]:
accidents = accidents.drop(columns=["Vitesse max autorisée", "Nombre de voies"])
accidents.shape

(54401, 13)

In [32]:
accidents = accidents.rename(columns={"Situation de l\x92accident": "Situation de l'accident"})

In [33]:
print(accidents.columns.tolist())

['Département', 'Agglomération', 'Luminosité', 'Météo (conditions atmos.)', 'Type de collision', 'Catégorie de route', 'Régime de circulation', 'État de la surface', 'Infrastructure', "Situation de l'accident", 'Gravité (label)', 'Vitesse_num', 'Voies_num']


In [34]:
for col in accidents.columns:
    if "Non renseigné" in accidents[col].values:
        print(col, accidents[col].value_counts()["Non renseigné"])

Type de collision 6
Régime de circulation 2971
État de la surface 2
Infrastructure 448


In [35]:
accidents = accidents.replace("Non renseigné", "Inconnu")

In [36]:
for col in accidents.columns:
    if "Non renseigné" in accidents[col].values:
        print(col, accidents[col].value_counts()["Non renseigné"])
print("Il n'y a plus de colonne Non renseigné.")

Il n'y a plus de colonne Non renseigné.


In [37]:
accidents.dtypes

Département                      str
Agglomération                    str
Luminosité                       str
Météo (conditions atmos.)        str
Type de collision                str
Catégorie de route               str
Régime de circulation            str
État de la surface               str
Infrastructure                   str
Situation de l'accident          str
Gravité (label)                  str
Vitesse_num                  float64
Voies_num                    float64
dtype: object

In [38]:
accidents.columns.tolist()

['Département',
 'Agglomération',
 'Luminosité',
 'Météo (conditions atmos.)',
 'Type de collision',
 'Catégorie de route',
 'Régime de circulation',
 'État de la surface',
 'Infrastructure',
 "Situation de l'accident",
 'Gravité (label)',
 'Vitesse_num',
 'Voies_num']

In [39]:
accidents["Département"].value_counts()

Département
75     4191
93     2640
92     2484
13     2120
94     1963
       ... 
90       61
978      37
977      22
986       6
975       2
Name: count, Length: 107, dtype: int64

In [40]:
accidents["Département"].unique()

<StringArray>
[ '70',  '21',  '15',  '14',  '13',   '7',   '6',  '75',  '94',  '51',
 ...
  '24',  '48',  '58',   '9',  '52',  '23', '977', '986', '978', '975']
Length: 107, dtype: str

In [41]:
region_to_depts = {
    'Île de France': ['75', '77', '78', '91', '92', '93','94', '95'],
    'Auvergne-Rhône-Alpes':     ['01', '03', '07', '15', '26', '38', '42', '43', '63', '69', '73', '74'],
    'Bretagne':                 ['22', '29', '35', '56'],
    'Bourgogne-Franche-Comté':  ['21', '25', '39', '58', '70', '71', '89', '90'],  
    'Normandie':                ['14', '27', '50', '61', '76'],  
    'Hauts-de-France':          ['02', '59', '60', '62', '80'],  
    'Grand Est':                ['08', '10', '51', '52', '54', '55', '57', '67', '68', '88'],  
    'Pays de la Loire':         ['44', '49','53', '72', '85'],  
    'Centre-Val de Loire':      ['18', '28', '36', '37', '41', '45'],  
    'Nouvelle-Aquitaine':       ['16', '17', '19', '23', '24', '33', '40', '47', '64', '79', '86', '87'],  
    'Occitanie':                ['09', '11', '12', '30', '31', '32', '34', '46', '48', '65', '66', '81', '82'],  
    'PACA':                     ['04', '05', '06', '13', '83', '84'],  
    'Corse':                    ['2A', '2B'],
    'Outre-mer':                ['971', '972', '973', '974', '975', '976', '977', '978', '986', '987','988']
}

In [42]:
dept_to_region = {}
for region, depts in region_to_depts.items():
    for dept in depts:
        dept_to_region[dept] = region
print(dept_to_region)

{'75': 'Île de France', '77': 'Île de France', '78': 'Île de France', '91': 'Île de France', '92': 'Île de France', '93': 'Île de France', '94': 'Île de France', '95': 'Île de France', '01': 'Auvergne-Rhône-Alpes', '03': 'Auvergne-Rhône-Alpes', '07': 'Auvergne-Rhône-Alpes', '15': 'Auvergne-Rhône-Alpes', '26': 'Auvergne-Rhône-Alpes', '38': 'Auvergne-Rhône-Alpes', '42': 'Auvergne-Rhône-Alpes', '43': 'Auvergne-Rhône-Alpes', '63': 'Auvergne-Rhône-Alpes', '69': 'Auvergne-Rhône-Alpes', '73': 'Auvergne-Rhône-Alpes', '74': 'Auvergne-Rhône-Alpes', '22': 'Bretagne', '29': 'Bretagne', '35': 'Bretagne', '56': 'Bretagne', '21': 'Bourgogne-Franche-Comté', '25': 'Bourgogne-Franche-Comté', '39': 'Bourgogne-Franche-Comté', '58': 'Bourgogne-Franche-Comté', '70': 'Bourgogne-Franche-Comté', '71': 'Bourgogne-Franche-Comté', '89': 'Bourgogne-Franche-Comté', '90': 'Bourgogne-Franche-Comté', '14': 'Normandie', '27': 'Normandie', '50': 'Normandie', '61': 'Normandie', '76': 'Normandie', '02': 'Hauts-de-France',

In [43]:
accidents["Département"] = accidents["Département"].str.zfill(2)

In [44]:
accidents["Région"] = accidents["Département"].map(dept_to_region)

In [45]:
accidents["Région"].value_counts()

Région
Île de France              15365
Auvergne-Rhône-Alpes        5854
PACA                        4732
Nouvelle-Aquitaine          4272
Occitanie                   3794
Outre-mer                   3344
Grand Est                   3280
Hauts-de-France             2704
Normandie                   2374
Pays de la Loire            2360
Bretagne                    2340
Centre-Val de Loire         1719
Bourgogne-Franche-Comté     1691
Corse                        572
Name: count, dtype: int64

In [46]:
accidents["Région"].isna().sum()

np.int64(0)

In [47]:
codes_manquants = accidents.loc[accidents["Région"].isna(), "Département"].unique()
print(codes_manquants)

<StringArray>
[]
Length: 0, dtype: str


In [48]:
accidents = accidents.drop(columns=["Département"])

In [49]:
accidents.shape

(54401, 13)

In [50]:
print(accidents.columns.to_list())

['Agglomération', 'Luminosité', 'Météo (conditions atmos.)', 'Type de collision', 'Catégorie de route', 'Régime de circulation', 'État de la surface', 'Infrastructure', "Situation de l'accident", 'Gravité (label)', 'Vitesse_num', 'Voies_num', 'Région']


In [51]:
X = accidents.drop(columns=['Gravité (label)'])
y = accidents['Gravité (label)']

In [52]:
X.shape

(54401, 12)

In [53]:
y.shape

(54401,)

In [54]:
X_lab_encoded = X.copy()

In [55]:
for col in X_lab_encoded:
    if X_lab_encoded[col].dtypes == 'str':
        le = LabelEncoder()
        X_lab_encoded[col] = le.fit_transform(X_lab_encoded[col])

In [56]:
X_lab_encoded.dtypes

Agglomération                  int64
Luminosité                     int64
Météo (conditions atmos.)      int64
Type de collision              int64
Catégorie de route             int64
Régime de circulation          int64
État de la surface             int64
Infrastructure                 int64
Situation de l'accident        int64
Vitesse_num                  float64
Voies_num                    float64
Région                         int64
dtype: object

In [57]:
X_oh_encoding = X.copy()

In [58]:
cat_cols = [col for col in X_oh_encoding.columns if X_oh_encoding[col].dtypes == 'str']

In [59]:
X_oh_encoding = pd.get_dummies(X_oh_encoding, columns=cat_cols)

In [60]:
X_oh_encoding.shape

(54401, 81)

In [61]:
y.value_counts()

Gravité (label)
Blessé léger          34743
Blessé hospitalisé    16432
Tué                    3226
Name: count, dtype: int64

In [62]:
gravite_mapping = {
    'Blessé léger': 0,
    'Blessé hospitalisé': 1,
    'Tué': 2
}

In [63]:
y = y.map(gravite_mapping)

In [64]:
y.value_counts()

Gravité (label)
0    34743
1    16432
2     3226
Name: count, dtype: int64

In [65]:
os.makedirs("donnees/clean", exist_ok=True)

X_lab_encoded.to_csv("donnees/clean/X_lab_encoded.csv", index=False)
X_oh_encoding.to_csv("donnees/clean/X_oh_encoding.csv", index=False)
y.to_csv("donnees/clean/y_gravite.csv", index=False)